# Satellite Parking Spot Detection — YOLOv8m-OBB
Train `yolov8m-obb.pt` on aerial parking images and extract center points of each slot.

In [ ]:
!pip install ultralytics -q

In [ ]:
import torch
import ultralytics

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
ultralytics.checks()

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

## Upload Dataset
Upload your `dataset.zip`.  Expected layout:
Labels must be in **YOLO-OBB format**: `class cx1 cy1 cx2 cy2 cx3 cy3 cx4 cy4`

In [ ]:
from google.colab import files
import zipfile, yaml
from pathlib import Path

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall("/content/")

DATA_YAML = list(Path("/content").rglob("data.yaml"))[0]
DATASET   = DATA_YAML.parent
print(f"Dataset root: {DATASET}")

# patch path for colab
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = str(DATASET)

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

CLASS_NAMES = list(cfg['names'].values())
print(f"Classes: {CLASS_NAMES}")

for split in ['train', 'val']:
    n = len(list((DATASET/split/'images').glob('*.*')))
    print(f"  {split}: {n} images")

## Visualize Annotations

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

COLORS = {0: (72, 199, 142), 1: (255, 107, 107)}

def draw_obb(img_path, lbl_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if not Path(lbl_path).exists():
        return img
    for line in open(lbl_path).read().strip().splitlines():
        vals  = list(map(float, line.split()))
        cls   = int(vals[0])
        pts   = (np.array(vals[1:9]).reshape(4,2) * [w, h]).astype(int)
        cx, cy = pts[:,0].mean(), pts[:,1].mean()
        cv2.polylines(img, [pts], True, COLORS[cls], 2)
        cv2.circle(img, (int(cx), int(cy)), 5, (255,255,255), -1)
        cv2.circle(img, (int(cx), int(cy)), 5, COLORS[cls], 2)
    return img

train_imgs = sorted((DATASET/'train'/'images').glob('*.*'))[:6]
fig, axes  = plt.subplots(2, 3, figsize=(16, 9))

for ax, p in zip(axes.flatten(), train_imgs):
    lbl = DATASET/'train'/'labels'/(p.stem+'.txt')
    ax.imshow(draw_obb(p, lbl))
    ax.set_title(p.name, fontsize=8)
    ax.axis('off')

for ax in axes.flatten()[len(train_imgs):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m-obb.pt")

model.train(
    data      = str(DATA_YAML),
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,         
    device    = DEVICE,
    optimizer = 'AdamW',
    degrees   = 45.0,      
    patience  = 20,
    amp       = True,
    project   = '/content/runs',
    name      = 'parking_obb',
    plots     = True,
)

WEIGHTS = Path('/content/runs/parking_obb/weights/best.pt')
print(f"Best weights: {WEIGHTS}")

## Evaluate

In [ ]:
best = YOLO(str(WEIGHTS))
m    = best.val(data=str(DATA_YAML), device=DEVICE)

print(f"mAP@50    : {m.box.map50:.4f}")
print(f"mAP@50-95 : {m.box.map:.4f}")
print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")

In [ ]:
import pandas as pd

df  = pd.read_csv('/content/runs/parking_obb/results.csv')
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, title in zip(axes,
    ['train/box_loss', 'metrics/mAP50(B)', 'metrics/precision(B)'],
    ['Box Loss', 'mAP@50', 'Precision']):
    ax.plot(df['epoch'], df[col])
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Inference + Center Points

In [ ]:
def get_slots(model, img_path, conf=0.25, iou=0.45):
    """Run inference and return annotated image + list of slot dicts."""
    img  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    vis  = img.copy()
    pred = model.predict(str(img_path), conf=conf, iou=iou, device=DEVICE, verbose=False)[0]
    slots = []

    if pred.obb is None:
        return vis, slots

    for i, (corners, cls, conf_val) in enumerate(zip(
        pred.obb.xyxyxyxy.cpu().numpy(),
        pred.obb.cls.cpu().numpy().astype(int),
        pred.obb.conf.cpu().numpy()
    )):
        cx, cy = corners[:,0].mean(), corners[:,1].mean()
        color  = COLORS.get(cls, (255,255,0))
        pts    = corners.astype(int)

        cv2.polylines(vis, [pts], True, color, 2)
        cv2.circle(vis, (int(cx), int(cy)), 5, (255,255,255), -1)
        cv2.circle(vis, (int(cx), int(cy)), 5, color, 2)
        cv2.putText(vis, f"#{i}", tuple(pts[0]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

        slots.append({
            'id'        : i,
            'class'     : CLASS_NAMES[cls],
            'confidence': round(float(conf_val), 3),
            'center_x'  : round(float(cx), 1),
            'center_y'  : round(float(cy), 1),
        })

    return vis, slots

In [ ]:
val_imgs = sorted((DATASET/'val'/'images').glob('*.*'))[:6]
all_data = []

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, p in zip(axes.flatten(), val_imgs):
    vis, slots = get_slots(best, p)
    n_e = sum(1 for s in slots if s['class'] == 'empty_slot')
    n_o = sum(1 for s in slots if s['class'] == 'occupied_slot')
    ax.imshow(vis)
    ax.set_title(f"{p.name}\nEmpty: {n_e}  Occupied: {n_o}", fontsize=8)
    ax.axis('off')
    for s in slots:
        s['image'] = p.name
        all_data.append(s)

for ax in axes.flatten()[len(val_imgs):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df_out = pd.DataFrame(all_data)[['image','id','class','confidence','center_x','center_y']]
print(df_out.to_string(index=False))

df_out.to_csv('/content/slot_centers.csv', index=False)
print("\nSaved → /content/slot_centers.csv")

## Download Results

In [ ]:
import shutil
from google.colab import files

shutil.copy(WEIGHTS, '/content/best.pt')
files.download('/content/best.pt')
files.download('/content/slot_centers.csv')

## Run on Custom Image (Optional)

In [ ]:
uploaded = files.upload()
img_path = Path(list(uploaded.keys())[0])

vis, slots = get_slots(best, img_path)

plt.figure(figsize=(12, 8))
plt.imshow(vis)
plt.title(f"{img_path.name} — {len(slots)} slots detected")
plt.axis('off')
plt.show()

for s in slots:
    print(f"Slot {s['id']:>3} | {s['class']:<15} | conf: {s['confidence']} | center: ({s['center_x']}, {s['center_y']})")